# Example 2: Transient heat transfer in 1D

This notebook walks through [`eg_2.py`](eg_2.py). See [`README.md`](README.md) for the
full problem statement.

## The problem

The rod from Example 1, but now in **time**. It starts cold ($T = 0$) and the hot left
end drives heat in:

$$ \frac{\partial T}{\partial t} - \nabla\cdot(k\,\nabla T) = 0, \qquad
T(0, t) = 100, \quad T(1, t) = 0, \quad T(x, 0) = 0. $$

As $t$ grows the solution relaxes towards the straight line of Example 1.

## What is new

Everything from Example 1, plus the four ingredients of a transient solve: a
**previous-step function** `u_n`, a **time-derivative term** in the weak form, a
**time loop**, and **writing results to file**.

## 1. Backends and imports

The same backends as before (`mpi4py` for parallelism, `petsc4py` for solvers), plus
two new tools for a time-dependent run:

- **`tqdm`** for a progress bar over the time steps,
- **`dolfinx.io.VTXWriter`** to write the solution at each step to a `.bp` file you can
  open in ParaView.

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc

import dolfinx
import basix
import numpy as np
import tqdm.autonotebook
from dolfinx import fem
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.mesh import create_mesh
import ufl

from dolfinx.io import VTXWriter

## 2. The mesh

Unchanged from Example 1: a uniform 1D interval mesh on $(0, 1)$.

In [ ]:
# 100 points, 99 interval cells, on the domain (0, 1)
indices = np.linspace(0, 1, num=100)
gdim, shape, degree = 1, "interval", 1
domain = ufl.Mesh(basix.ufl.element("Lagrange", shape, degree, shape=(gdim,)))
mesh_points = np.reshape(indices, (len(indices), 1))
indexes = np.arange(mesh_points.shape[0])
cells = np.stack((indexes[:-1], indexes[1:]), axis=-1)
my_mesh = create_mesh(comm=MPI.COMM_WORLD, cells=cells, x=mesh_points, e=domain)
fdim = my_mesh.topology.dim - 1

## 3. Function space and functions

The new ingredient is **`u_n`**, a second function that stores the solution at the
**previous** time step. Both `u` (current, unknown) and `u_n` (known) live in the same
space. `u_n` starts at its default value of zero, which is exactly our initial
condition $T(x, 0) = 0$.

In [ ]:
V = fem.functionspace(my_mesh, ("Lagrange", 1))
u = fem.Function(V)        # current step (unknown)
u_n = fem.Function(V)      # previous step (known); starts at 0
v = ufl.TestFunction(V)

## 4. Boundary conditions

Unchanged: the two ends are pinned for all time, `T = 100` on the left and `T = 0` on
the right.

In [ ]:
dofs_L = fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0))
dofs_R = fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], indices[-1]))
bc_left = fem.dirichletbc(fem.Constant(my_mesh, PETSc.ScalarType(100)), dofs_L, V)
bc_right = fem.dirichletbc(fem.Constant(my_mesh, PETSc.ScalarType(0)), dofs_R, V)
bcs = [bc_left, bc_right]

## 5. The variational form, now with time

We discretise the time derivative with **backward Euler**:

$$ \frac{\partial T}{\partial t} \approx \frac{u - u_n}{\Delta t}. $$

The weak form is the steady diffusion term from Example 1 **plus** this time term:

$$ F = \int_\Omega k\,\nabla u \cdot \nabla v \; dx
     + \int_\Omega \frac{u - u_n}{\Delta t}\, v \; dx = 0. $$

In [ ]:
k = 0.1
dt = 0.1
F = ufl.dot(k * ufl.grad(u), ufl.grad(v)) * ufl.dx
F += ((u - u_n) / dt) * v * ufl.dx

## 6. The solver

Identical to the earlier examples: a `NonlinearProblem` driven by PETSc's SNES
(Newton) solver, configured with a direct LU solve (MUMPS) at each step. The trailing
loop clears the options out of the global PETSc database so they do not leak into
other solvers. See Example 1 for the line-by-line explanation.

In [ ]:
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_stol": np.sqrt(np.finfo(dolfinx.default_real_type).eps) * 1e-2,
    "snes_atol": 1e-10,
    "snes_rtol": 1e-10,
    "snes_max_it": 30,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}
solver = NonlinearProblem(
    F,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="festim_solver",
)

# clear the options out of the global PETSc database
snes = solver.solver
prefix = snes.getOptionsPrefix()
opts = PETSc.Options()
for key in petsc_options.keys():
    del opts[f"{prefix}{key}"]

## 7. Solve in a time loop

Now we step through time. At each step we:

1. **solve** for `u` at the new time,
2. **copy** `u` into `u_n` so it becomes the previous step,
3. **write** the result to the `.bp` file,
4. **advance** `t` and update the progress bar.

`VTXWriter` accumulates every step into `ht_transient.bp`, which you can open in
ParaView and play back as an animation.

In [ ]:
writer = VTXWriter(MPI.COMM_WORLD, "ht_transient", u, "BP5")

final_time = 10
t = 0
progress = tqdm.autonotebook.tqdm(
    desc="Solving heat transfer problem", total=final_time, unit_scale=True
)
while t < final_time:
    solver.solve()
    u_n.x.array[:] = u.x.array
    writer.write(t)

    t += dt
    progress.update(dt)

writer.close()

## Recap

You added time stepping to the Example 1 skeleton: a previous-step function,
a backward-Euler time term, a solve loop, and file output. Open `ht_transient.bp` in
[ParaView](https://www.paraview.org/) to watch the rod heat up towards the Example 1
steady state.